# 08: Box certificates on calibration split 1

Design decisions
- Everything comes from the frozen configurations of notebook 05: the configuration hash, the model hash, the prior hash, the split hash and the hashes of the explained subsamples are checked before any calibration row is read.
- Candidate sets (frozen M, feature orders, search budget and inflation rule) are built for every explained patient of calibration split 2 and of the test split from their features only; their outcomes are not read here. The candidate cache is saved on Drive every 500 patients, so an interrupted run resumes.
- Calibration split 1 is opened only after that. For every distinct candidate box: n_B records inside, k_B of them whose recorded outcome differs from the box class, the frozen prior pi(B), and the bound U(k_B, n_B; delta_1 pi(B)) by the inverse survival function (Eq. 6). The bound does not depend on the targets, which enter only at release.
- A box can be released only if its bound is at most the loosest target on the frozen grids (0.10 for negative-outcome claims, 0.80 for enrichment claims). Every such box is re-checked by the second exact soundness check; a box it cannot finish is never released, and a counterexample stops the notebook. Z3 checks a random sample of these boxes.
- Outputs: the box table on Drive with its SHA-256 recorded in results/08_certificates.json, and a summary per task in results/08_certificates_summary.csv.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask
from scax.explain.candidates import CandidateGenerator

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
assert REC05['written_before_calibration_split_1']
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'; CERT_DIR.mkdir(parents=True, exist_ok=True)


def load_task(task):
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == FT[task].cfg['split_sha256'], 'split differs from the frozen one'
    return X.to_numpy(dtype=float), y, code


def generator(ft):
    c = ft.cfg['candidates']
    gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
    assert gen.settings['inflation'] == c['inflation']
    return gen


for t in TASKS:
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| model', FT[t].cfg['model']['sha256'][:16],
          '| prior', FT[t].cfg['prior']['sha256'][:16], '| alpha_sel', FT[t].targets['alpha_sel'])

## Candidate sets for the explained patients (features only)

In [ ]:
CAND = {}
for task in TASKS:
    ft = FT[task]
    Xa, y, code = load_task(task)
    del y                                   # outcomes of explained patients are not used in this notebook
    rows = np.r_[ft.cal2_rows, ft.test_rows]
    C = ft.grid.cells(Xa[rows]); yhat = ft.pred.predict_cells(C)
    gen = generator(ft)
    cache = CERT_DIR / f'{task}__candidates.pkl'
    if gen.load(cache):
        print(f'{task}: {len(gen.cache)} cells loaded from Drive')
    t0 = time.time()
    for i in range(len(C)):
        if gen.cell_key(C[i]) not in gen.cache:
            gen.candidates(C[i], int(yhat[i]))
            if gen.oracle.stats.cells % 500 == 0:
                gen.save(cache)
                print(f'  {task}: {gen.oracle.stats.cells} new cells, {time.time() - t0:.0f}s')
    gen.save(cache)
    gen.oracle.memo.clear()                 # free memory before the next task; the candidate sets stay
    CAND[task] = (gen, C, yhat, rows)
    secs = list(gen.seconds.values())
    print(f'{task}: {len(rows)} explained patients, {len(gen.cache)} distinct cells, median {np.median(secs):.2f}s per cell')

## Certificates on calibration split 1

In [ ]:
from scax.certify.counting import box_counts
from scax.certify.certificate import upper_bounds

BOXES = {}
for task in TASKS:
    ft = FT[task]; gen, C, yhat, rows = CAND[task]
    Xa, y, code = load_task(task)
    cal1 = splits.indices(code, 'cal1')
    C1, y1 = ft.grid.cells(Xa[cal1]), y[cal1]
    boxes = sorted({(c.key, c.y) for k in {gen.cell_key(r) for r in C} for c in gen.cache[k]})
    keys = [k for k, _ in boxes]; cls = np.array([cl for _, cl in boxes])
    t0 = time.time()
    n_B, k_B = box_counts(keys, cls, C1, y1)
    log_pi = ft.log_prior(keys)
    U = upper_bounds(k_B, n_B, np.log(ft.targets['delta_1']) + log_pi)
    loosest = {0: max(ft.targets['alpha_0_grid']), 1: round(1 - min(ft.targets['beta_grid']), 10)}
    eligible = U <= np.where(cls == 1, loosest[1], loosest[0])
    BOXES[task] = pd.DataFrame({'key': [json.dumps(k) for k in keys], 'box_class': cls, 'size': [len(k) for k in keys],
                                'n_B': n_B, 'k_B': k_B, 'log_prior': log_pi, 'U': U, 'eligible': eligible})
    print(f'{task}: calibration split 1 has {len(cal1)} records ({int(y1.sum())} with the outcome); '
          f'{len(keys)} boxes, {int(eligible.sum())} could be released ({time.time() - t0:.0f}s)')

## Second soundness check on every box that could be released

In [ ]:
from scax.explain.verify import verify_boxes, z3_sample

VERIFY = {}
for task in TASKS:
    ft = FT[task]; df = BOXES[task]
    el = df[df.eligible]
    boxes = [(tuple(tuple(t) for t in json.loads(k)), int(c)) for k, c in zip(el.key, el.box_class)]
    t0 = time.time()
    verdicts, times = verify_boxes(ft.grid, boxes)
    z3res, zt = z3_sample(ft.grid, [b for b in boxes if verdicts[b] == 'sound'], 60, SEED, timeout_ms=10000, time_budget_s=150)
    df['verified'] = 'not needed'
    df.loc[df.eligible, 'verified'] = [verdicts[b] for b in boxes]
    VERIFY[task] = {'eligible_boxes': len(boxes), 'sound': sum(v == 'sound' for v in verdicts.values()),
                    'unknown': sum(v == 'unknown' for v in verdicts.values()), 'median_ms': 1000 * float(np.median(times)) if times else None,
                    'z3_sample': z3res, 'seconds': time.time() - t0}
    print(task, VERIFY[task])

## Box table and summary

In [ ]:
SUMMARY, HASHES = [], {}
for task in TASKS:
    ft = FT[task]; df = BOXES[task]; gen, C, yhat, rows = CAND[task]
    f = CERT_DIR / f'{task}__boxes.parquet'
    df.to_parquet(f, index=False)
    HASHES[task] = sha256_file(f)
    a0, a1 = ft.targets['alpha_0'], ft.targets['alpha_1']
    passing = df[(df.verified == 'sound') & (df.U <= np.where(df.box_class == 1, a1, a0))]
    pass_keys = set(zip(passing.key, passing.box_class))
    has = np.array([any((json.dumps(c.key), c.y) in pass_keys for c in gen.cache[gen.cell_key(r)]) for r in C])
    n_cal2 = len(ft.cal2_rows)
    for cl in (0, 1):
        d = df[df.box_class == cl]
        SUMMARY.append({'task': task, 'box_class': cl, 'boxes': len(d), 'eligible': int(d.eligible.sum()),
                        'passing_main_target': int(((d.verified == 'sound') & (d.U <= (a1 if cl else a0))).sum()),
                        'median_n_B': float(d.n_B.median()), 'median_size': float(d['size'].median()),
                        'median_U': float(d.U.median()),
                        'cal2_explained_with_a_passing_box': float(has[:n_cal2][yhat[:n_cal2] == cl].mean()) if (yhat[:n_cal2] == cl).any() else None,
                        'test_explained_with_a_passing_box': float(has[n_cal2:][yhat[n_cal2:] == cl].mean()) if (yhat[n_cal2:] == cl).any() else None})
SUM = pd.DataFrame(SUMMARY)
SUM.to_csv(f'{paths.RESULTS}/08_certificates_summary.csv', index=False)
print(SUM.round(4).to_string(index=False))
write_json(f'{paths.RESULTS}/08_certificates.json', {
    'box_table_sha256': HASHES, 'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'verification': VERIFY,
    'candidate_cells': {t: len(CAND[t][0].cache) for t in TASKS},
    'median_seconds_per_cell': {t: float(np.median(list(CAND[t][0].seconds.values()))) for t in TASKS}})

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "08: candidate sets for the explained patients and box certificates on calibration split 1 for tasks B and A, second check on every releasable box"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)